In [1]:
import torch
from training_loop import training_loop
from extract_embedding import get_train_test_loaders
from qwen_tts.core.models import BasicSpeakerEncoder
import optuna

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



********
********
 


2026-04-22 03:46:16.805728070 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
# Setup the data loaders
train_loader, test_loader = get_train_test_loaders()


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB
Loading Qwen3-TTS model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 36157.79it/s]


Qwen3 speaker encoder loaded on cuda
train-clean-100 already extracted
dev-clean already extracted
Loaded 12526 samples
Loaded 1953 samples
Train: 12526 samples, 391 batches
Test:  1953 samples, 62 batches


In [3]:

def objective(trial, train_loader, test_loader):
    # 1. Define the hyperparameter search space
    # Optuna will sample values from these ranges for each trial
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    alpha = trial.suggest_float("loss_alpha", 0.1, 0.9) 
    
    # Assemble the dictionary for the training loop
    hyperparams = {
        "num_epochs": 20,       # Keep epochs lower for faster sweeps (tune as needed)
        "save_every": 20,       # Prevent excessive checkpoint saving during sweeps
        "lr": lr,
        "loss_alpha": alpha,
        "weight_decay": weight_decay,
        "model_name": f"BasicSpeakerEncoder_trial_{trial.number}"
    }

    # 2. Setup Device and model
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    # Setup the model
    model = BasicSpeakerEncoder().to(device)
    
        
    # 3. Execute the training loop
    # We catch the returned best test cosine similarity
    best_test_cosine = training_loop(model, train_loader, test_loader, device, hyperparams)
    
    # Optuna will try to MAXIMIZE this returned value
    return best_test_cosine


In [ ]:
study = optuna.create_study(direction="minimize", study_name="Speaker_Encoder_Distillation")
study.optimize(lambda trial: objective(trial, train_loader, test_loader), n_trials=20)
    
print("\n=================================")
print("Best Trial Found:")
print(f"Best Test Loss: {study.best_trial.value:.4f}")
print("Optimal Hyperparameters: ")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")